In [3]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/nutrition/daily_food_nutrition_dataset.csv", on_bad_lines='skip')

print(df.shape)
print(df.columns)
df.head()

(645, 12)
Index(['Food_Item', 'Category', 'Calories (kcal)', 'Protein (g)',
       'Carbohydrates (g)', 'Fat (g)', 'Fiber (g)', 'Sugars (g)',
       'Sodium (mg)', 'Cholesterol (mg)', 'Meal_Type', 'Water_Intake (ml)'],
      dtype='object')


,Food_Item,Category,Calories (kcal),Protein (g),Carbohydrates (g),Fat (g),Fiber (g),Sugars (g),Sodium (mg),Cholesterol (mg),Meal_Type,Water_Intake (ml)
0,Scrambled Eggs (2 large),Protein/Dairy,180,12.0,2.0,14.0,0.0,1.0,180,370,Breakfast,250
1,Whole Wheat Toast (1 slice),Grain,80,4.0,14.0,1.0,2.0,2.0,140,0,Breakfast,0
2,Coffee (black),Beverage,5,0.3,0.0,0.1,0.0,0.0,5,0,Breakfast,0
3,Banana,Fruit,105,1.3,27.0,0.4,3.1,14.0,1,0,Breakfast,0
4,Grilled Chicken Salad,Meal/Protein,350,30.0,10.0,20.0,5.0,4.0,400,80,Lunch,500


In [6]:
import numpy as np

# simulate 100 users, 30 days
num_users = 100
days = 30

data = []

for user in range(num_users):
    for day in range(days):
        sample = df.sample(3)  # 3 meals/day

        data.append({
            "user_id": user,
            "day": day,
            "calories": sample["Calories (kcal)"].sum(),
            "protein": sample["Protein (g)"].sum(),
            "carbs": sample["Carbohydrates (g)"].sum(),
            "fat": sample["Fat (g)"].sum()
        })

user_df = pd.DataFrame(data)

In [7]:
print(user_df.head())
print(user_df.shape)

   user_id  day  calories  protein  carbs   fat
0        0    0       157      1.3   19.4   9.3
1        0    1       720     17.0   85.0  36.0
2        0    2       248      4.0   26.5  14.0
3        0    3       257      8.7   29.2  12.2
4        0    4       646     22.6   71.0  16.0
(3000, 6)


In [10]:
# Rename base columns first
user_df = user_df.rename(columns={
    "calories": "Calories",
    "protein": "Protein",
    "carbs": "Carbs",
    "fat": "Fat"
})

**FEATURE ENGINEERING**

**1. ratio feature**

In [11]:
user_df["Protein_Ratio"] = user_df["Protein"] / (user_df["Calories"] + 1)
user_df["Carb_Ratio"] = user_df["Carbs"] / (user_df["Calories"] + 1)
user_df["Fat_Ratio"] = user_df["Fat"] / (user_df["Calories"] + 1)

per meal feature


In [12]:
user_df["Protein_Per_Meal"] = user_df["Protein"] / 3
user_df["Carb_Per_Meal"] = user_df["Carbs"] / 3
user_df["Fat_Per_Meal"] = user_df["Fat"] / 3

3. Rolling Features (Time Behavior)**bold text**

In [13]:
user_df["Calories_Rolling_Mean"] = user_df.groupby("user_id")["Calories"].transform(lambda x: x.rolling(7).mean())
user_df["Protein_Rolling_Mean"] = user_df.groupby("user_id")["Protein"].transform(lambda x: x.rolling(7).mean())

**4. Trend Features**

In [14]:
user_df["Calories_Diff"] = user_df.groupby("user_id")["Calories"].diff()
user_df["Protein_Diff"] = user_df.groupby("user_id")["Protein"].diff()

user_df["Calories_Trend"] = user_df["Calories_Diff"].apply(lambda x: 1 if x > 0 else 0)

**5. Density Feature**

In [15]:
user_df["Calorie_Density"] = user_df["Calories"] / (
    user_df["Protein"] + user_df["Carbs"] + user_df["Fat"] + 1
)

In [16]:
user_df = user_df.dropna()

In [17]:
feature_cols = [
    "Calories", "Protein", "Carbs", "Fat",
    "Protein_Ratio", "Carb_Ratio", "Fat_Ratio",
    "Calories_Rolling_Mean", "Protein_Rolling_Mean",
    "Calories_Diff", "Protein_Diff",
    "Calorie_Density"
]

**rebuild sequence**

In [19]:
values = user_df[feature_cols].values

STEP 1 — Create feature matrix

In [ ]:
values = user_df[feature_cols].values

STEP 2 — Build sequences

In [21]:
import numpy as np

def build_sequences_per_user(df, seq_len=7):
    X, y = [], []

    users = df["user_id"].unique()

    for user in users:
        user_data = df[df["user_id"] == user].sort_values("day")

        vals = user_data[feature_cols].values

        for i in range(len(vals) - seq_len):
            X.append(vals[i:i+seq_len])
            y.append(vals[i+seq_len])

    return np.array(X), np.array(y)

In [22]:
X, y = build_sequences_per_user(user_df, seq_len=7)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1700, 7, 12)
y shape: (1700, 12)


STEP 3 — Normalize **Features**

In [23]:
from sklearn.preprocessing import StandardScaler
import numpy as np

scaler = StandardScaler()

# reshape for scaling
num_samples, seq_len, num_features = X.shape

X_reshaped = X.reshape(-1, num_features)
X_scaled = scaler.fit_transform(X_reshaped)

X = X_scaled.reshape(num_samples, seq_len, num_features)

y = scaler.transform(y)

**Train/Test Split**

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True
)

step 5 - Convert to PyTorch

In [25]:
import torch

X_train = torch.tensor(X_train).float()
y_train = torch.tensor(y_train).float()

X_test = torch.tensor(X_test).float()
y_test = torch.tensor(y_test).float()

STEP 6 — Improved LSTM Model

In [26]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        return self.fc(out)

STEP 7 — Training Loop

In [27]:
model = LSTMModel(input_size=X.shape[2])

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

epochs = 10

for epoch in range(epochs):
    model.train()

    pred = model(X_train)
    loss = loss_fn(pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # evaluation
    model.eval()
    with torch.no_grad():
        val_pred = model(X_test)
        val_loss = loss_fn(val_pred, y_test)

    print(f"Epoch {epoch+1}: Train Loss={loss.item():.4f}, Val Loss={val_loss.item():.4f}")

Epoch 1: Train Loss=0.9902, Val Loss=0.9879
Epoch 2: Train Loss=0.9843, Val Loss=0.9838
Epoch 3: Train Loss=0.9814, Val Loss=0.9797
Epoch 4: Train Loss=0.9766, Val Loss=0.9758
Epoch 5: Train Loss=0.9726, Val Loss=0.9719
Epoch 6: Train Loss=0.9690, Val Loss=0.9680
Epoch 7: Train Loss=0.9655, Val Loss=0.9642
Epoch 8: Train Loss=0.9613, Val Loss=0.9604
Epoch 9: Train Loss=0.9579, Val Loss=0.9565
Epoch 10: Train Loss=0.9535, Val Loss=0.9527


STEP 8 — Evaluate

In [28]:
from sklearn.metrics import mean_squared_error

preds = model(X_test).detach().numpy()
actual = y_test.numpy()

mse = mean_squared_error(actual, preds)
print("Test MSE:", mse)

Test MSE: 0.952711820602417


**STEP 1 — Positional Encoding**

In [29]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

STEP 2 — Transformer Model

In [30]:
class TransformerModel(nn.Module):
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()

        self.input_fc = nn.Linear(input_size, d_model)

        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc_out = nn.Linear(d_model, input_size)

    def forward(self, x):
        x = self.input_fc(x)
        x = self.pos_encoder(x)

        x = self.transformer(x)

        x = x[:, -1, :]   # last timestep
        return self.fc_out(x)

**STEP 3 — Initialize Model**

In [31]:
model = TransformerModel(input_size=X.shape[2])

**STEP 4 — Training Loop**

In [32]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

epochs = 10

for epoch in range(epochs):
    model.train()

    pred = model(X_train)
    loss = loss_fn(pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_pred = model(X_test)
        val_loss = loss_fn(val_pred, y_test)

    print(f"Epoch {epoch+1}: Train={loss.item():.4f}, Val={val_loss.item():.4f}")

Epoch 1: Train=1.3022, Val=1.3213
Epoch 2: Train=1.3353, Val=1.0128
Epoch 3: Train=1.0326, Val=0.9175
Epoch 4: Train=0.9418, Val=0.9221
Epoch 5: Train=0.9442, Val=0.9148
Epoch 6: Train=0.9372, Val=0.8888
Epoch 7: Train=0.9112, Val=0.8621
Epoch 8: Train=0.8886, Val=0.8441
Epoch 9: Train=0.8733, Val=0.8340
Epoch 10: Train=0.8628, Val=0.8284


**1. Add normalization check**

In [33]:
print(X.mean(), X.std())

-3.1896800793533236e-15 1.0


**2. Train longer**

In [34]:
epochs = 20

**3. Add learning rate scheduler**

In [35]:
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [36]:
scheduler.step()

/tmp/ipykernel_1927/3814219595.py:1: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


4. Check prediction **quality**

In [37]:
preds = model(X_test).detach().numpy()
actual = y_test.numpy()

print("Pred:", preds[0])
print("Actual:", actual[0])

Pred: [-0.30448532 -0.23472315 -0.34817642 -0.04637209 -0.079509    0.0965264
 -0.07903539  0.3827448   0.11646568  0.4985304   0.89818406  0.15462947]
Actual: [ 0.49571052 -0.26162016  0.2647942   0.88491946 -0.57339686 -0.3039538
  0.77030545  0.17944184 -0.69625527  0.86190313  0.61981463  0.21443465]


**Recommendation Layer**

STEP 1 — Take prediction

In [59]:
# STEP 1 — Prediction
pred_scaled = model(X_test)[0].detach().numpy()

# STEP 2 — Convert to real values
pred_real = scaler.inverse_transform([pred_scaled])[0]

# STEP 3 — Recommendation logic starts HERE 👇

print("Real Prediction:", pred_real)

Real Prediction: [3.66978789e+02 1.54109921e+01 3.70375038e+01 1.87899595e+01
 3.93674672e-02 1.17297422e-01 4.02079129e-02 4.73619242e+02
 1.96200088e+01 1.69399601e+02 1.91285510e+01 5.40661249e+00]


**STEP 2 — Compare with food dataset**

In [60]:
food_features = df[["Calories (kcal)", "Protein (g)", "Carbohydrates (g)", "Fat (g)"]].values

In [61]:
filtered_df = df[
    (df["Calories (kcal)"] > 50) &
    (df["Protein (g)"] > 2)
]

In [55]:
food_features = filtered_df[[
    "Calories (kcal)", "Protein (g)", "Carbohydrates (g)", "Fat (g)"
]].values

STEP 3 — Find closest food

In [81]:
from sklearn.metrics.pairwise import euclidean_distances
import numpy as np

# =========================
# STEP 1 — Prediction
# =========================
pred_scaled = model(X_test)[0].detach().numpy()
pred_real = scaler.inverse_transform([pred_scaled])[0]

target = pred_real[:4]  # [Calories, Protein, Carbs, Fat]

# =========================
# STEP 2 — Filter foods
# =========================
filtered_df = df[
    (df["Calories (kcal)"] > 50) &
    (df["Protein (g)"] > 2)
]

# remove drinks
filtered_df = filtered_df[
    ~filtered_df["Food_Item"].str.contains("coffee|latte|soda", case=False)
]

food_features = filtered_df[[
    "Calories (kcal)",
    "Protein (g)",
    "Carbohydrates (g)",
    "Fat (g)"
]].values

# =========================
# STEP 3 — Candidate pool
# =========================
distances = euclidean_distances([target], food_features)

# larger pool for diversity
idx = distances.argsort()[0][:100]
candidates = filtered_df.iloc[idx]

# =========================
# STEP 4 — Build meal
# =========================

# ---- Stage 1: Base food ----
best_idx = distances.argsort()[0][0]
base_food = filtered_df.iloc[best_idx]

selected = [base_food["Food_Item"]]

total = np.array([
    base_food["Calories (kcal)"],
    base_food["Protein (g)"],
    base_food["Carbohydrates (g)"],
    base_food["Fat (g)"]
])

# ---- Stage 2: Try smart side (light + optimized) ----
side_df = filtered_df[
    (filtered_df["Calories (kcal)"] < 150) &

    # avoid pure fat / useless items
    (filtered_df["Protein (g)"] > 2) &

    # ensure meaningful carbs (veggies/grains)
    (filtered_df["Carbohydrates (g)"] > 5)
]
best_side = None
best_score = float("inf")

for _, row in side_df.iterrows():

    food_vec = np.array([
        row["Calories (kcal)"],
        row["Protein (g)"],
        row["Carbohydrates (g)"],
        row["Fat (g)"]
    ])

    new_total = total + food_vec

    # control overshoot
    if new_total[0] > target[0] * 1.2:
        continue

    score = np.linalg.norm(target - new_total)

    if score < best_score:
        best_score = score
        best_side = row

# ---- Add optimized side ----
if best_side is not None:
    selected.append(best_side["Food_Item"])
    total += np.array([
        best_side["Calories (kcal)"],
        best_side["Protein (g)"],
        best_side["Carbohydrates (g)"],
        best_side["Fat (g)"]
    ])

# =========================
# STEP 5 — Output
# =========================
print("\n🎯 Target Nutrition:", target)
print("\n🍽️ Meal Plan:", selected)
print("\n📊 Total Nutrition:", total)


🎯 Target Nutrition: [366.97878947  15.41099212  37.03750378  18.78995951]

🍽️ Meal Plan: ['Manicotti (2 shells cheese)', 'Steamed Broccoli (1 cup)']

📊 Total Nutrition: [435.   23.7  46.2  18.6]


**STEP 4 — Add Nutrition Balance Filter**

In [67]:
filtered_df = filtered_df[
    (filtered_df["Carbohydrates (g)"] > 5) &
    (filtered_df["Fat (g)"] > 2)
]

STEP 5 — Weighted Distance

In [68]:
import numpy as np

weights = np.array([1.0, 2.0, 1.0, 1.0])  # protein more important

distances = np.sqrt(((food_features - pred[:4])**2 * weights).sum(axis=1))